In [ ]:
import os
# from nltk import word_tokenize
# from nltk.corpus import stopwords
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('punkt_tab')

# arabic_stopwords = set(stopwords.words('arabic'))
# import numpy as np
# import pandas as pd
# text1="يعيش محمد في جمهورية مصر العربية"
# text2="يعيش احمد في المملكة العربية السعودية"
# text3="يتقابلان مرة كل عام في مصر"

# # text1="The car is driven on the road"
# # text2="The truck is driven on the highway"

# def tf_idf(sent):
#   vocab=[]
#   for i in sent:
#     for j in word_tokenize(i):
#       if j not in vocab:
#         vocab.append(j)
#   words_count=np.zeros(shape=(len(sent),len(vocab)))
#   for i in range(len(sent)):
#     for j in vocab:
#       if j in word_tokenize(sent[i]):
#         words_count[i][vocab.index(j)]+=1
#   each_word_count=np.sum(words_count,axis=0)
#   tf=words_count/np.sum(words_count,axis=1,keepdims=True)
#   idf=np.zeros(shape=(len(sent),len(vocab)))
#   df_tf=pd.DataFrame(tf,columns=vocab)
#   c=0
#   for i in words_count:
#     x=0
#     for j in i:
#       if j!=0:
#         idf[c][x]=np.log10(len(sent)/each_word_count[x])
#       x+=1
#     c+=1
#   df_idf=pd.DataFrame(idf,columns=vocab)
#   words_count_df=pd.DataFrame(words_count,columns=vocab)
#   tfidf_df=df_tf*df_idf
#   return df_tf,df_idf,words_count_df,tfidf_df

# tf,idf,words_count,tf_idf=tf_idf([text1,text2,text3])


# # tf,idf,words_count,tf_idf

In [ ]:
# from google.colab import auth
# auth.authenticate_user()

# # import google.auth
# from google.auth import default
# creds, _ = default()

# # Now you can use 'creds' in your Google API calls
# import io

# import google.auth
# from googleapiclient.discovery import build
# from googleapiclient.errors import HttpError
# from googleapiclient.http import MediaIoBaseDownload


# def download_file(creds,file_id,file_name):

#   service = build("drive", "v3", credentials=creds)
#   request = service.files().get_media(fileId=file_id)

#   # Save the file locally
#   with open(file_name, 'wb') as file:
#       downloader = MediaIoBaseDownload(file, request)
#       done = False
#       while not done:
#           status, done = downloader.next_chunk()
#           print(f"Download {int(status.progress() * 100)}%")

# download_file(creds,"140Ylpyzyzz7A1TtX2vx18wkC9ZLJiBlG","test.txt")

In [ ]:
!pip install google-api-python-client
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
from bs4 import BeautifulSoup
import requests
import re
warnings.filterwarnings('ignore')
from googleapiclient.discovery import build

api_key = os.getenv("API_KEY")
youtube_service = build('youtube', 'v3', developerKey=api_key)

def get_vedio_comments(youtubeAPI,vedio_url):
  pattern = r"(?:v=|\/v\/|embed\/|youtu\.be\/|\/watch\?v=)([A-Za-z0-9_-]{11})"
  match = re.search(pattern, vedio_url)
  video_id=""
  if match:
    video_id = match.group(1)
    # print(f"Video ID: {video_id}")
  else:
    print("Check the link")
    return
  request=youtube_service.commentThreads().list(part="snippet,replies",videoId=video_id,maxResults=20)
  respone=request.execute()

  user_name_and_comment={}
  comments=respone['items']

for i in comments:
  user_name=i['snippet']['topLevelComment']['snippet']['authorDisplayName']
  comment=i['snippet']['topLevelComment']['snippet']['textDisplay']
  if user_name not in user_name_and_comment.keys():
    user_name_and_comment[user_name]=[comment]
  else:
    user_name_and_comment[user_name].append(comment)

  return user_name_and_comment

user_name_and_comment=get_vedio_comments(youtube_service,"https://www.youtube.com/watch?v=cu-81Lgqres&ab_channel=%D9%85%D8%AD%D9%85%D8%AF%D9%85%D8%B5%D8%B7%D9%81%D9%89%D8%B9%D8%A7%D8%B4%D9%82%D8%A7%D9%84%D9%81%D9%86")

# all_comnets=""
# for val in user_name_and_comment.values():
#   for i in val:
#     all_comnets+=i+' '


# from wordcloud import WordCloud,ImageColorGenerator
# from PIL import Image

# img=Image.open('/content/egypt_flag.jpg')
# mask=np.array(img)


# # Generate a word cloud image
# img_colours=ImageColorGenerator(mask)
# wordcloud = WordCloud(font_path='/content/Zumurud.ttf',width=800, height=400, background_color='white',mask=mask,color_func=img_colours).generate(all_comnets)
# plt.figure(figsize=(10,8))
# plt.imshow(wordcloud, interpolation='bilinear')
# plt.axis("off")
# plt.show()

In [ ]:
# !pip install camel-tools
from camel_tools.utils.dediac import dediac_ar

def clean_text(text):
  cleaned_text = re.sub(r"[\u06D6-\u06D7-\u06D8-\u06D9-\u06DA-\u06DB-\u06DC-\u06E0-\u06E1-\u06E2-\u06E3-\u06E4-\u06E5-\u06E6-\u06E7-\u06E8-\u06E9-\u06EA-\u06EB-\u06EC-\u06ED\s]",
                        " ", text)
  cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
  cleaned_text = dediac_ar(cleaned_text)
  return cleaned_text

from wordcloud import WordCloud,ImageColorGenerator
from PIL import Image

quran_words=pd.read_csv('/content/data.csv')
txt=quran_words.text.values

final_txt=""
for i in txt:
  final_txt+=clean_text(i)+' '

img=Image.open("/content/egypt_flag.jpg")
mask=np.array(img)
img_colours = ImageColorGenerator(mask)

wordcloud=WordCloud(width=1920, height=1080,font_path="/content/DTHULUTH-II-1.ttf",background_color='black',colormap="spring").generate(final_txt) # mask=mask,color_func=img_colours
plt.figure(figsize=(10,8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

In [ ]:
!pip install datasets
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

In [ ]:
ds = load_dataset("arbml/ArSAS")
ds=ds['train'].train_test_split(test_size=0.25)

In [ ]:
ds=ds.remove_columns(column_names=['#Tweet_ID','Topic','Sentiment_label_confidence','Speech_act_label','Speech_act_label_confidence'])

In [ ]:
lbl_names=ds['train'].features['label'].names
lable2id={}
id2lable={}
for c,i in enumerate(lbl_names):
  lable2id[i]=c
  id2lable[c]=i

In [ ]:
toknizer=AutoTokenizer.from_pretrained("Alibaba-NLP/gte-multilingual-reranker-base")

In [ ]:
def text_toknizeation(batch):
  text=batch['Tweet_text']
  toknized_text=toknizer(text).input_ids
  return {'input_ids': toknized_text}

toknized_text=ds.map(text_toknizeation,batched=True,remove_columns=['Tweet_text'])

In [ ]:
!pip install evaluate

from transformers import DataCollatorWithPadding
import evaluate

data_collator = DataCollatorWithPadding(tokenizer=toknizer)
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "Alibaba-NLP/gte-multilingual-reranker-base", num_labels=4, id2label=id2lable,label2id=lable2id,trust_remote_code=True,ignore_mismatched_sizes=True
)

In [ ]:
from transformers import TrainingArguments,Trainer

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    gradient_accumulation_steps=16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=toknized_text["train"],
    eval_dataset=toknized_text["test"],
    processing_class=toknizer,
    data_collator=data_collator,
    # compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
toknizer.save_pretrained("/content/Text_analysis_model")
model.save_pretrained("/content/Text_analysis_model")

In [ ]:
from huggingface_hub import HfApi, HfFolder

HfFolder.save_token("[REDACTED_SECRET]")

repo_id = "abosalah/arabic_Text_analysis_model"  # Specify a unique name for your model repo
api = HfApi()
api.create_repo(repo_id=repo_id, private=True)


model.push_to_hub(repo_id)
toknizer.push_to_hub(repo_id)


In [ ]:
import torch
import tensorflow as tf

def text_predict(text,model_folder):
  toknizer2=AutoTokenizer.from_pretrained(model_folder)
  model2=AutoModelForSequenceClassification.from_pretrained(model_folder,trust_remote_code=True)
  encoded_text=toknizer2(text,return_tensors="pt")
  with torch.no_grad():
    res=model2(**encoded_text)
  final_res=tf.nn.softmax(res.logits.numpy()).numpy()
  res_class=id2lable[np.argmax(final_res)]
  acc=np.max(final_res)
  return res_class,acc

t="طبعا 25 يناير ثورة محترمة ونفخر بها كان فيها عملاء وخونه واصحاب اجندات ايوة كان فيها زيها زى اى ثورة بيحاول اصحاب المصالح الركوب عليها وده حصل فعلا من الاخوان وتم تصحيح المسار بى 30 / 6 واللى يقول كلام غير ده يبقى كداب لان دى الحقيقة المجردة من اى مصالح او اطماع"
model_folder="/content/Text_analysis_model"

text_predict(t,model_folder)


In [ ]:
testo = load_dataset("arbml/ArSAS")

In [ ]:
rand_num=np.random.randint(0,len(testo['train']))
t=testo['train'][rand_num]['Tweet_text']

print(t,id2lable[testo['train'][rand_num]['label']])

print(text_predict(t,model_folder))

In [ ]:
!pip install google-api-python-client
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
from bs4 import BeautifulSoup
import requests
import re
warnings.filterwarnings('ignore')
from googleapiclient.discovery import build

api_key = os.getenv("API_KEY")
youtube_service = build('youtube', 'v3', developerKey=api_key)

def get_vedio_comments(youtubeAPI,vedio_url):
  pattern = r"(?:v=|\/v\/|embed\/|youtu\.be\/|\/watch\?v=)([A-Za-z0-9_-]{11})"
  match = re.search(pattern, vedio_url)
  video_id=""
  if match:
    video_id = match.group(1)
    # print(f"Video ID: {video_id}")
  else:
    print("Check the link")
    return
  request=youtube_service.commentThreads().list(part="snippet,replies",videoId=video_id,maxResults=20)
  respone=request.execute()

  user_name_and_comment={}
  comments=respone['items']

  for i in comments:
    user_name=i['snippet']['topLevelComment']['snippet']['authorDisplayName']
    comment=i['snippet']['topLevelComment']['snippet']['textDisplay']
    if user_name not in user_name_and_comment.keys():
      user_name_and_comment[user_name]=[comment]
    else:
      user_name_and_comment[user_name].append(comment)
  return user_name_and_comment

user_name_and_comment=get_vedio_comments(youtube_service,"https://www.youtube.com/watch?v=cu-81Lgqres&ab_channel=%D9%85%D8%AD%D9%85%D8%AF%D9%85%D8%B5%D8%B7%D9%81%D9%89%D8%B9%D8%A7%D8%B4%D9%82%D8%A7%D9%84%D9%81%D9%86")

# all_comnets=""
# for val in user_name_and_comment.values():
#   for i in val:
#     all_comnets+=i+' '


# from wordcloud import WordCloud,ImageColorGenerator
# from PIL import Image

# img=Image.open('/content/egypt_flag.jpg')
# mask=np.array(img)


# # Generate a word cloud image
# img_colours=ImageColorGenerator(mask)
# wordcloud = WordCloud(font_path='/content/Zumurud.ttf',width=800, height=400, background_color='white',mask=mask,color_func=img_colours).generate(all_comnets)
# plt.figure(figsize=(10,8))
# plt.imshow(wordcloud, interpolation='bilinear')
# plt.axis("off")
# plt.show()

In [ ]:
x=list(user_name_and_comment.values())

x[2]

In [ ]:
text_predict(x[2][0],model_folder)